## Environment

In [2]:
# Quant data policy: this repo reads Google Drive data; downloads run from note.
import sys
from pathlib import Path
_QUANT_ROOT = Path("/Users/xinc/GitHub/Quant")
if str(_QUANT_ROOT) not in sys.path:
    sys.path.insert(0, str(_QUANT_ROOT))
import cloud_data
from datetime import datetime
from pathlib import Path
import os
import sys
import pandas as pd
REPO_ROOT = Path("/Users/xinc/GitHub/Quant")
NOTE_ROOT = Path("/Users/xinc/GitHub/note")
sys.path.extend([str(NOTE_ROOT), str(REPO_ROOT), os.getcwd()])
%load_ext autoreload
%autoreload 2
from module.options.option_tools import compute_iv
from analyzer import StrategyConfig, TXAnalyzer
from cloud_data import (
    MARKET_FEAR_GREED,
    MARKET_MOVE_DAILY,
    MARKET_SOX_DAILY,
    TW_OPTIONS_INSTITUTION_DAY,
    TW_OPTIONS_INSTITUTION_NIGHT,
    TW_OPTIONS_SETTLE_TXO,
    TW_STOCK_OTC_MARGIN_BALANCE,
    read_frame,
    read_tx_futures,
)
START = "2020-01-01"
END = datetime.now().strftime("%Y-%m-%d")
TRAIN_RATIO = 0.7
VALIDATION_RATIO = 0.05
SETTLEMENT_PATH = TW_OPTIONS_SETTLE_TXO

## Base Market Data

In [3]:
analyzer = TXAnalyzer(read_tx_futures(START, END))
analyzer.session_alignment_report()
split_dates = TXAnalyzer.split_periods(
    analyzer.display_df().index,
    start=START,
    train_ratio=TRAIN_RATIO,
    validation_ratio=VALIDATION_RATIO,
)
TRAIN_END = split_dates["train_end"]
VALIDATION_START = split_dates["validation_start"]
VALIDATION_END = split_dates["validation_end"]
TEST_START = split_dates["test_start"]
pd.Series(split_dates, name="date")

train_end          2024-08-02
validation_start   2024-08-05
validation_end     2024-11-25
test_start         2024-11-26
Name: date, dtype: datetime64[ns]

## Factor Discovery

### Price and Calendar

In [8]:
training_analyzer = analyzer.for_period(end=TRAIN_END)
training_analyzer.daily_ret()
training_analyzer.monthly_ret(mode='benchmark')
training_analyzer.indicator_weekday_stats()
training_analyzer.indicator_ma_divergence(window=25)
training_analyzer.indicator_hist_vol(window=40)
training_analyzer.indicator_night_ret(window=3)
training_analyzer.indicator_night_ret_divergence(window=3)
training_analyzer.indicator_gap_days(after_holiday=False)
training_analyzer.indicator_gap_days(after_holiday=True)


### Option Positioning

In [ ]:
training_analyzer = analyzer.for_period(end=TRAIN_END)
training_analyzer.indicator_opt_position(indicator="Foreign_Opt_Signal", trading_session="day")


### Option Implied Volatility

In [ ]:
training_analyzer = analyzer.for_period(end=TRAIN_END)
training_analyzer.indicator_option_iv(trading_session="day")


### Sentiment

In [ ]:
training_analyzer = analyzer.for_period(end=TRAIN_END)
training_analyzer.indicator_fear_greed(trading_session="day")
training_analyzer.indicator_fear_greed(trading_session="night")


### Global Market Factors

In [ ]:
training_analyzer = analyzer.for_period(end=TRAIN_END)
training_analyzer.indicator_move(trading_session="day")
training_analyzer.indicator_move(trading_session="night")
training_analyzer.indicator_sox(trading_session="day")


### Rates and Margin

In [9]:
training_analyzer = analyzer.for_period(end=TRAIN_END)
training_analyzer.indicator_otc_margin_growth()
training_analyzer.indicator_otc_margin_growth_divergence(window=40)